# Baseline 2 — Exp18 Multi-Seed Robustness Check (Exp15 setting)

**Purpose:** Verify whether Exp15's higher development HM (seen 72.52%, unseen 4.25%, HM ≈ 8.03%) persists with different random seeds. One training run per `Run All`.

- Fixed development split (`splits_v2`), train-fit 6,905 / seen-val 695 / unseen-val 400; no official-test data.
- Exact Exp15 configuration: ResNet-50 V2, crop/disease cross-entropy, natural shuffle, task weights **1.0 / 1.5**, 224 input, batch 16, 4 workers, AdamW, cosine schedule, patience 15, max 30 epochs.
- Run **seed 123** first. Then set `EXPERIMENT_TO_RUN = "exp18_seed2026"`, restart kernel, Run All again. They save in independent folders.
- The already completed Exp15 (seed 42) is the first reference run; this notebook never re-trains seed 42.
- Compare per-seed development HM, crop/disease accuracy and error patterns; report mean, sample SD, and the small unseen validation denominator (400 images).
- These are **multiple training seeds on one fixed development split**, NOT independent dataset splits or independent official test results. Same development set has informed many prior experiments.


## 1. Select exactly ONE seed, check files / CUDA and run preflight

Leave `EXPERIMENT_TO_RUN="exp18_seed123"` for the first run. After completion change the selector to `"exp18_seed2026"` and use **Restart Kernel → Run All**. Do not overwrite existing experiment output folders.


In [ ]:
from pathlib import Path
from dataclasses import dataclass
import copy
import gc
import json
import math
import platform
import random
import tempfile
import time

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader
import torchvision
from torchvision import transforms as T
from torchvision.transforms import InterpolationMode

from model_resnet50 import MultiTaskResNet50, load_resnet50_checkpoint as load_checkpoint
from evaluate import (DPDDataset, MEAN, STD, evaluate_loader, generalisation_metrics,
                      validate_development_splits, validation_transform,
                      make_candidates, require_candidate_coverage,
                      sha256_file, synthetic_self_test)

PROJECT_DIR = Path.cwd()
SPLIT_DIR = PROJECT_DIR / "splits_v2"
DATASET_ROOT = PROJECT_DIR.parent / "DPD_Small_dataset"
IMAGE_ROOT = DATASET_ROOT
OUTPUT_ROOT = PROJECT_DIR / "outputs"

# ONE experiment per Run All; change only this selector for a later, optional run.
EXPERIMENT_TO_RUN = "exp18_seed123"  # later: "exp18_seed2026"

BASE_CONFIG = {
    "backbone": "resnet50", "pretrained_weights": "IMAGENET1K_V2",
    "crop_classes": 55, "disease_classes": 175, "image_size": 224,
    "batch_size": 16, "max_epochs": 30, "patience": 15, "min_delta": 1e-6,
    "backbone_lr": 1e-5, "head_lr": 1e-3, "crop_dropout": 0.3, "disease_dropout": 0.2,
    "weight_decay": 1e-3, "optimizer": "AdamW", "scheduler": "CosineAnnealingLR",
    "crop_loss_weight": 1.0, "disease_loss_weight": 1.0,
    "seed": 42, "num_workers": 4, "amp": True,
    "crop_scale": [0.85, 1.0], "flip_probability": 0.5, "vertical_flip_probability": 0.5,
    "rotation_degrees": 10, "brightness": 0.15, "contrast": 0.15,
    "interpolation": "bilinear_resize_nearest_rotation", "mean": list(MEAN), "std": list(STD),
    "candidate_mode": "all_pairs", "candidate_catalog": None,
}
EXPERIMENTS = {
    "exp18_seed123": {
        "changed_parameter": "Exp15 identical config, ONLY random seed 42 -> 123; same splits",
        "seed": 123, "sampling_method": "natural_shuffle", "sampling_alpha": 0.0,
        "disease_loss_method": "cross_entropy", "disease_cb_beta": None,
        "crop_loss_weight": 1.0, "disease_loss_weight": 1.5,
    },
    "exp18_seed2026": {
        "changed_parameter": "Exp15 identical config, ONLY random seed 42 -> 2026; same splits",
        "seed": 2026, "sampling_method": "natural_shuffle", "sampling_alpha": 0.0,
        "disease_loss_method": "cross_entropy", "disease_cb_beta": None,
        "crop_loss_weight": 1.0, "disease_loss_weight": 1.5,
    },
}
if EXPERIMENT_TO_RUN not in EXPERIMENTS:
    raise ValueError(f"Choose one of {list(EXPERIMENTS)}")
CONFIG = {**BASE_CONFIG, **EXPERIMENTS[EXPERIMENT_TO_RUN], "experiment_id": EXPERIMENT_TO_RUN}
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EXP09_DIR = OUTPUT_ROOT / "b2_resnet50_exp09_b1matched_seed42"
EXP15_DIR = OUTPUT_ROOT / "exp15_disease_weight15"
EXP10_DIR = OUTPUT_ROOT / "exp10"
EXPERIMENT_DIR = OUTPUT_ROOT / EXPERIMENT_TO_RUN
RUN_AUTO_AUDIT = True
print("PyTorch:", torch.__version__, "torchvision:", torchvision.__version__)
print("Selected:", EXPERIMENT_TO_RUN)
print(json.dumps(CONFIG, indent=2))
if DEVICE.type != "cuda":
    raise RuntimeError("CUDA GPU unavailable. STOP; do not train for hours on CPU.")
print("GPU:", torch.cuda.get_device_name(0))
if not IMAGE_ROOT.is_dir() or not SPLIT_DIR.is_dir():
    raise FileNotFoundError("Dataset or splits_v2 folder is missing; ensure notebook is in Baseline2 project root")
if EXPERIMENT_DIR.exists():
    raise FileExistsError(f"Output already exists: {EXPERIMENT_DIR}. Review old run; never silently overwrite it.")
gpu_free_bytes, gpu_total_bytes = torch.cuda.mem_get_info()
print(f"CUDA free VRAM: {gpu_free_bytes / 2**30:.2f}/{gpu_total_bytes / 2**30:.2f} GiB")
if gpu_free_bytes < 3*2**30:
    raise RuntimeError("Less than 3 GiB free VRAM. Close stale processes and restart kernel first.")
assert CONFIG["sampling_method"] == "natural_shuffle" and CONFIG["disease_loss_method"] == "cross_entropy"
assert CONFIG["crop_loss_weight"] == 1.0 and CONFIG["disease_loss_weight"] == 1.5
if not (EXP15_DIR / "validation_metrics.json").is_file():
    raise FileNotFoundError("Exp15 reference results missing: " + str(EXP15_DIR))
print("PREFLIGHT PASSED: ONE independent Exp18 seed will train. Exp15 remains untouched.")


## 2. Verify development splits and class mappings

Same train/validation split and label mapping as Exp09; train 6,905, seen-val 695, unseen-val 400. Validation never used for weight adjustment or training. No official test.


In [ ]:
frames, split_report = validate_development_splits(SPLIT_DIR)
print(json.dumps(split_report, indent=2))
canonical_metadata = pd.concat(frames.values(), ignore_index=True)
class_mappings = {
    task: {str(int(row[0])): str(row[1]) for row in
           canonical_metadata[[f"{task}_id", f"{task}_name"]].drop_duplicates().itertuples(index=False, name=None)}
    for task in ("crop", "disease", "composition")
}
composition_mapping = canonical_metadata[
    ["composition_id", "crop_id", "disease_id", "composition_name"]
].drop_duplicates().sort_values("composition_id").to_dict(orient="records")
print("Original IDs retained; no new class numbering.")


## 3. Augmentation, evaluation transforms and 9,625 candidate pairs

Identical to Exp09/Exp10: training RandomResizedCrop + flips + small rotation/color jitter; evaluation Resize(256) + CenterCrop(224); ImageNet normalization. Changing these would confound the intervention.

In [ ]:
def training_transform(config):
    return T.Compose([
        T.RandomResizedCrop(config["image_size"], scale=tuple(config["crop_scale"]),
                            interpolation=InterpolationMode.BILINEAR),
        T.RandomHorizontalFlip(p=config["flip_probability"]),
        T.RandomVerticalFlip(p=config["vertical_flip_probability"]),
        T.RandomRotation(config["rotation_degrees"], interpolation=InterpolationMode.NEAREST),
        T.ColorJitter(brightness=config["brightness"], contrast=config["contrast"]),
        T.ToTensor(), T.Normalize(MEAN, STD),
    ])

def evaluation_transform(image_size):
    resized = round(image_size * 256 / 224)
    return T.Compose([
        T.Resize(resized, interpolation=InterpolationMode.BILINEAR),
        T.CenterCrop(image_size),
        T.ToTensor(), T.Normalize(MEAN, STD),
    ])


candidates = make_candidates(CONFIG["candidate_mode"], CONFIG["candidate_catalog"])
require_candidate_coverage(candidates, list(frames.values()))
print(f"Candidate mode: {CONFIG['candidate_mode']}; {len(candidates):,} global candidate pairs")
if CONFIG["candidate_mode"] == "all_pairs":
    print("PROVISIONAL: confirm the same candidate policy with Baseline 1 before comparing results.")

## 4. Reproducibility: fixed split and seeds 42 / 123 / 2026

The dataset partition is **identical** across all seeds; only initial weights of the heads, DataLoader shuffle/augmentation sequences, etc. change. No WeightedRandomSampler, no class-balanced loss. Distinct output folders protect earlier weights. `num_workers=4` for all runs; batch 16, CUDA AMP.


In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def make_loader(dataset, config, shuffle=False, sampler=None):
    workers = int(config["num_workers"])
    if workers < 0:
        raise ValueError("num_workers must be >= 0")
    generator = torch.Generator().manual_seed(config["seed"])
    if shuffle and sampler is not None:
        raise ValueError("Cannot enable both shuffle and sampler")
    kwargs = dict(batch_size=config["batch_size"], shuffle=shuffle, sampler=sampler,
                  num_workers=workers, pin_memory=DEVICE.type == "cuda",
                  generator=generator, drop_last=False)
    if workers > 0:
        kwargs["prefetch_factor"] = 2
        kwargs["persistent_workers"] = True
    return DataLoader(dataset, **kwargs)


@dataclass
class EarlyStopping:
    patience: int = 5
    min_delta: float = 1e-6
    best: float = -math.inf
    bad_epochs: int = 0

    def update(self, value):
        if not math.isfinite(value):
            raise ValueError("Nonfinite validation harmonic mean")
        improved = value > self.best + self.min_delta
        if improved:
            self.best = value
            self.bad_epochs = 0
        else:
            self.bad_epochs += 1
        return improved, self.bad_epochs >= self.patience


def atomic_checkpoint(payload, path):
    path = Path(path)
    temporary = path.with_suffix(".tmp")
    torch.save(payload, temporary)
    temporary.replace(path)


def environment_details():
    return {"python": platform.python_version(), "torch": str(torch.__version__),
            "torchvision": str(torchvision.__version__), "numpy": np.__version__,
            "pandas": pd.__version__, "device": str(DEVICE),
            "gpu": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else None,
            "cuda_runtime": torch.version.cuda}


def make_tempered_disease_sampler(training_frame, alpha=0.25, seed=42):
    """Compute per-image weights exclusively from the existing train-fit labels."""
    if not (0 <= alpha <= 1):
        raise ValueError("alpha must be in [0,1]")
    frequency = training_frame["disease_id"].value_counts()
    if (frequency <= 0).any() or len(training_frame) != 6905:
        raise ValueError("Unexpected train-fit disease frequency / split size")
    per_image_counts = training_frame["disease_id"].map(frequency)
    weights = torch.as_tensor(per_image_counts.to_numpy(dtype=float) ** (-alpha), dtype=torch.double)
    assert len(weights) == len(training_frame) and torch.isfinite(weights).all()
    sampler = torch.utils.data.WeightedRandomSampler(
        weights=weights, num_samples=len(training_frame), replacement=True,
        generator=torch.Generator().manual_seed(seed))
    # Approximate expected draws per label (not a guarantee about realised sampled counts).
    expected = (pd.DataFrame({"disease_id":frequency.index.astype(int),
                 "train_images":frequency.values.astype(int)}).sort_values("disease_id"))
    numerators = expected["train_images"].to_numpy(dtype=float) ** (1-alpha)
    expected["expected_draws_per_epoch"] = len(training_frame) * numerators / numerators.sum()
    return sampler, expected



## 5. CPU synthetic smoke test

Brief CPU-only shape, gradient and checkpoint tests. Real full training remains CUDA + AMP. No Stage 2 and no official test.


In [ ]:
def run_smoke_checks():
    synthetic_self_test()
    seed_everything(CONFIG["seed"])
    # Run synthetic smoke check on CPU so it cannot exhaust RTX 4050 VRAM.
    # Section 7 real training still runs on CUDA, with mixed precision.
    smoke_device = torch.device("cpu")
    smoke = MultiTaskResNet50(pretrained=False, crop_dropout=CONFIG["crop_dropout"], disease_dropout=CONFIG["disease_dropout"]).to(smoke_device)
    assert smoke.crop_head[1].in_features == smoke.disease_head[1].in_features == 2048
    assert not ({id(p) for p in smoke.crop_head.parameters()} &
                {id(p) for p in smoke.disease_head.parameters()})
    groups = smoke.parameter_groups()
    group_ids = [id(p) for group in groups for p in list(group["params"])]
    assert len(group_ids) == len(set(group_ids)) == len(list(smoke.parameters()))
    # Recreate groups because the generator above was consumed.
    optimizer = torch.optim.AdamW(smoke.parameter_groups(), weight_decay=1e-4)
    # Smoke check uses FP32 for stable gradient assertions; real training still uses CUDA AMP.
    scaler = torch.amp.GradScaler("cuda", enabled=False)
    images = torch.randn(2, 3, 64, 64, device=smoke_device)  # Synthetic check, small to reduce CPU load
    with torch.autocast(device_type=smoke_device.type, enabled=False):
        output = smoke(images)
        assert output["crop_logits"].shape == (2, 55)
        assert output["disease_logits"].shape == (2, 175)
        lc = nn.functional.cross_entropy(output["crop_logits"], torch.tensor([0, 54], device=smoke_device))
        ld = nn.functional.cross_entropy(output["disease_logits"], torch.tensor([0, 174], device=smoke_device))
    shared_parameter = next(smoke.backbone.parameters())
    for loss in (lc, ld):
        gradient = torch.autograd.grad(loss, shared_parameter, retain_graph=True)[0]
        assert torch.isfinite(gradient).all() and gradient.abs().sum() > 0
    loss = CONFIG["crop_loss_weight"] * lc + CONFIG["disease_loss_weight"] * ld
    assert torch.isfinite(loss) and CONFIG["disease_loss_weight"] > CONFIG["crop_loss_weight"]
    scaler.scale(loss).backward()
    for module in (smoke.backbone, smoke.crop_head, smoke.disease_head):
        grads = [p.grad for p in module.parameters() if p.grad is not None]
        assert grads and all(torch.isfinite(g).all() for g in grads)
    scaler.step(optimizer)
    scaler.update()
    smoke.eval()
    with torch.no_grad():
        before = smoke(images)["crop_logits"].cpu()
    with tempfile.TemporaryDirectory() as directory:
        path = Path(directory) / "synthetic.pt"
        atomic_checkpoint({"model_state": smoke.state_dict(), "config": CONFIG}, path)
        restored, _ = load_checkpoint(path, smoke_device)
        with torch.no_grad():
            after = restored(images)["crop_logits"].cpu()
        assert torch.allclose(before, after, atol=1e-6, rtol=1e-5)
    stopper = EarlyStopping()
    assert stopper.update(0) == (True, False)
    for _ in range(4):
        assert stopper.update(0) == (False, False)
    assert stopper.update(0) == (False, True)
    assert stopper.update(0.1) == (True, False)
    assert stopper.update(0.1 + 1e-8) == (False, False)
    del smoke, restored, images, output, loss, lc, ld, optimizer
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    print("CPU synthetic checks passed; actual Section 7 training remains CUDA + AMP.")

run_smoke_checks()

## 6. Exp15 task-loss training functions (unchanged)

Shared backbone and two heads trained jointly. `L = CE_crop + 1.5*CE_disease`. Each run saves the highest **development HM** checkpoint (not the final epoch). A later improvement is possible even if validation loss shows overfitting signals. Compare loss curves separately.


In [ ]:
def train_one_epoch(model, loader, optimizer, scaler, device, amp=True,
                    crop_loss_weight=1.0, disease_loss_weight=1.0, disease_class_weights=None):
    model.train()
    totals = np.zeros(3, dtype=float)
    n_images = 0
    crop_criterion = nn.CrossEntropyLoss()
    disease_criterion = nn.CrossEntropyLoss(weight=disease_class_weights)
    for batch in loader:
        images = batch["image"].to(device, non_blocking=True)
        crop = batch["crop"].to(device, non_blocking=True)
        disease = batch["disease"].to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        try:
            with torch.autocast(device_type=device.type, enabled=amp and device.type == "cuda"):
                result = model(images)
                crop_loss = crop_criterion(result["crop_logits"], crop)
                disease_loss = disease_criterion(result["disease_logits"], disease)
                total_loss = crop_loss_weight * crop_loss + disease_loss_weight * disease_loss
            if not torch.isfinite(total_loss):
                raise RuntimeError("Nonfinite training loss; stop and inspect the inputs/configuration.")
            scaler.scale(total_loss).backward()
            scaler.step(optimizer)
            scaler.update()
        except torch.cuda.OutOfMemoryError as error:
            raise RuntimeError("CUDA out of memory. Restart the kernel, explicitly reduce batch_size to 8, "
                               "record the change, and rerun as a new experiment. No setting was changed automatically.") from error
        count = images.shape[0]
        totals += count * np.array([total_loss.item(), crop_loss.item(), disease_loss.item()])
        n_images += count
    if not n_images:
        raise ValueError("Empty training loader")
    return dict(zip(("total_loss", "crop_loss", "disease_loss"), (totals / n_images).tolist()))


def plot_training_history(history, best_epoch, path):
    table = pd.DataFrame(history)
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    axes[0, 0].plot(table.epoch, table.train_total_loss, label="Training")
    # Evaluate-loader total_loss is unweighted; reweight validation task losses
    # to make this graph comparable with the weighted train_total_loss.
    cw = float(CONFIG["crop_loss_weight"])
    dw = float(CONFIG["disease_loss_weight"])
    for split,label in (("seen", "Seen validation"), ("unseen", "Unseen validation")):
        weighted = cw * table[f"{split}_crop_loss"] + dw * table[f"{split}_disease_loss"]
        axes[0, 0].plot(table.epoch, weighted, label=label)
    axes[0, 0].set(title=f"Task-weighted CE: crop {cw:g} + disease {dw:g}", ylabel="Mean weighted loss")
    axes[0, 1].plot(table.epoch, table.train_crop_loss, label="Crop")
    axes[0, 1].plot(table.epoch, table.train_disease_loss, label="Disease")
    axes[0, 1].set(title="Training task losses", ylabel="Mean loss")
    axes[1, 0].plot(table.epoch, table.seen_composition_top1 * 100, label="Seen-like")
    axes[1, 0].plot(table.epoch, table.unseen_composition_top1 * 100, label="Unseen-like")
    axes[1, 0].set(title="Composition Top-1 validation", ylabel="Accuracy (%)")
    axes[1, 1].plot(table.epoch, table.composition_harmonic_mean * 100, label="Harmonic mean")
    axes[1, 1].set(title="Development selection criterion", ylabel="Accuracy (%)")
    for ax in axes.flat:
        ax.axvline(best_epoch, linestyle="--", color="black", alpha=.5, label=f"Best epoch {best_epoch}")
        ax.set_xlabel("Epoch")
        ax.grid(alpha=.2)
        ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    return fig


def fit_stage1(config, train_loader, seen_loader, unseen_loader, candidate_pairs,
               metadata_report, mappings, composition_map, output_root=OUTPUT_ROOT):
    seed_everything(config["seed"])
    run_config = copy.deepcopy(config)
    run_config["candidate_catalog"] = (str(config["candidate_catalog"])
                                         if config["candidate_catalog"] is not None else None)
    # A failed pretrained download is an error, never a random-weight fallback.
    model = MultiTaskResNet50(crop_dropout=config["crop_dropout"],
                            disease_dropout=config["disease_dropout"], pretrained=True).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameter_groups(config["backbone_lr"], config["head_lr"]),
                                 weight_decay=config["weight_decay"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config["max_epochs"])
    amp_enabled = config["amp"] and DEVICE.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)
    stopper = EarlyStopping(config["patience"], config["min_delta"])
    output_dir = Path(output_root) / config["experiment_id"]
    output_dir.mkdir(parents=True, exist_ok=False)
    environment = environment_details()
    (output_dir / "config.json").write_text(json.dumps({"config": run_config, "environment": environment,
         "development_split": metadata_report, "class_mappings": mappings,
         "composition_mapping": composition_map,
         "candidate_pairs": candidate_pairs.cpu().tolist(),
         "candidate_policy_status": "provisional" if config["candidate_mode"] == "all_pairs" else "user-supplied group catalogue",
         "accuracy_units": "fractions in [0,1]"}, indent=2), encoding="utf-8")
    disease_class_weights = None
    if config["disease_loss_method"] == "effective_number":
        freq = np.bincount(frames["train_fit"]["disease_id"].to_numpy(dtype=int),
                            minlength=config["disease_classes"])
        if len(freq) != 175 or np.any(freq <= 0):
            raise ValueError("All 175 disease classes must be present in train_fit for this experiment")
        beta = float(config["disease_cb_beta"])
        if not (0.0 < beta < 1.0):
            raise ValueError("CB beta must be in (0, 1)")
        # Expm1 improves numerical stability when beta is near 1.
        effective_n = -np.expm1(freq * np.log(beta)) / (1.0-beta)
        weights = 1.0 / effective_n
        weights = weights / weights.mean()  # scale doesn't change PyTorch's weighted-mean CE
        disease_class_weights = torch.tensor(weights, device=DEVICE, dtype=torch.float32)
        pd.DataFrame({"disease_id": np.arange(175), "train_images": freq,
                      "effective_number": effective_n, "normalized_loss_weight": weights}).to_csv(
                          output_dir / "disease_effective_number_weights.csv", index=False)
        print("Class-balanced disease loss: beta=", beta,
              "weights min/max=", f"{weights.min():.3f}/{weights.max():.3f}")
    elif config["disease_loss_method"] != "cross_entropy":
        raise ValueError("Unsupported disease loss method")
    history = []
    best_epoch = None
    start = time.perf_counter()
    for epoch in range(1, config["max_epochs"] + 1):
        epoch_start = time.perf_counter()
        rates = [group["lr"] for group in optimizer.param_groups]
        training = train_one_epoch(model, train_loader, optimizer, scaler, DEVICE, amp_enabled,
                                   config["crop_loss_weight"], config["disease_loss_weight"], disease_class_weights)
        seen, _ = evaluate_loader(model, seen_loader, DEVICE, candidate_pairs, amp_enabled)
        unseen, _ = evaluate_loader(model, unseen_loader, DEVICE, candidate_pairs, amp_enabled)
        joint = generalisation_metrics(seen, unseen)
        row = {"epoch": epoch, "backbone_lr": rates[0], "head_lr": rates[1],
               **{f"train_{key}": value for key, value in training.items()},
               **{f"seen_{key}": value for key, value in seen.items()},
               **{f"unseen_{key}": value for key, value in unseen.items()},
               "composition_harmonic_mean": joint["composition_harmonic_mean"],
               "seen_unseen_gap": joint["seen_unseen_gap"],
               "epoch_seconds": time.perf_counter() - epoch_start}
        history.append(row)
        pd.DataFrame(history).to_csv(output_dir / "training_history.csv", index=False)
        improved, stop = stopper.update(joint["composition_harmonic_mean"])
        scheduler.step()
        if improved:
            best_epoch = epoch
            best_metrics = {"epoch": epoch, "seen": seen, "unseen": unseen, **joint}
            atomic_checkpoint({"stage": 1, "epoch": epoch, "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(), "scheduler_state": scheduler.state_dict(),
                "scaler_state": scaler.state_dict(), "config": run_config,
                "candidate_pairs": candidate_pairs.cpu().tolist(),
                "class_mappings": mappings, "composition_mapping": composition_map,
                "development_split": metadata_report, "environment": environment,
                "best_metrics": best_metrics, "parameter_count": sum(p.numel() for p in model.parameters())},
                output_dir / "best_development.pt")
        print(f"Epoch {epoch:02d} | loss {training['total_loss']:.4f} | "
              f"seen {seen['composition_top1']:.2%} | unseen {unseen['composition_top1']:.2%} | "
              f"HM {joint['composition_harmonic_mean']:.2%}" + (" * best" if improved else ""))
        if stop:
            print(f"Early stopping after {config['patience']} consecutive non-improving epochs.")
            break
    best = torch.load(output_dir / "best_development.pt", map_location=DEVICE, weights_only=True)
    model.load_state_dict(best["model_state"])
    model.eval()
    validation_metrics = {**best["best_metrics"], "epochs_run": len(history),
                          "training_seconds": time.perf_counter() - start,
                          "accuracy_units": "fractions in [0,1]"}
    (output_dir / "validation_metrics.json").write_text(json.dumps(validation_metrics, indent=2), encoding="utf-8")
    log = {"experiment_id": config["experiment_id"], "changed_parameter": config["changed_parameter"],
           "best_epoch": best_epoch, **generalisation_metrics(best["best_metrics"]["seen"], best["best_metrics"]["unseen"]),
           "candidate_mode": config["candidate_mode"], "checkpoint_path": str(output_dir / "best_development.pt"),
           "configuration": json.dumps(run_config, sort_keys=True)}
    pd.DataFrame([log]).to_csv(output_dir / "experiment_summary.csv", index=False)
    figure = plot_training_history(history, best_epoch, output_dir / "training_curves.png")
    plt.show()
    return model, history, validation_metrics, output_dir


## 7. Train ONE independent random seed (GPU + AMP)

Seed 123 or 2026 according to Section 1. Creates `outputs/<experiment_id>/`, stores `best_development.pt` and logs. 30 epochs maximum with patience 15.


In [ ]:
# Confirm the same split hashes and all-pair candidate policy were loaded.
frames, verified_report = validate_development_splits(SPLIT_DIR)
candidates = make_candidates(CONFIG["candidate_mode"], CONFIG["candidate_catalog"])
require_candidate_coverage(candidates, list(frames.values()))
for nm, frame in frames.items():
    if len(frame) != {"train_fit":6905,"val_seen":695,"val_unseen":400}[nm]:
        raise ValueError(f"Unexpected {nm} image count: {len(frame)}")

datasets = {name: DPDDataset(frame, IMAGE_ROOT,
    training_transform(CONFIG) if name=="train_fit" else evaluation_transform(CONFIG["image_size"]))
    for name,frame in frames.items()}
# Decode all metadata-listed development images before spending GPU time.
for dataset in datasets.values():
    for path in dataset.paths:
        with Image.open(path) as im: im.convert("RGB").load()
verified_report["image_paths_checked"] = True
verified_report["image_decoding_checked"] = True
sampler = None
exposure_table = None
assert CONFIG["sampling_method"] == "natural_shuffle"
print("Natural shuffle; disease task weight =", CONFIG["disease_loss_weight"],
      "; no WeightedRandomSampler / effective-number class weighting")
loaders = {name: make_loader(ds, CONFIG,
    shuffle=(name=="train_fit" and sampler is None),
    sampler=(sampler if name=="train_fit" else None)) for name,ds in datasets.items()}
seed_everything(CONFIG["seed"])
example = next(iter(loaders["val_seen"]))
assert tuple(example["image"].shape[1:]) == (3,224,224)
model, history, metrics, experiment_dir = fit_stage1(
    CONFIG,loaders["train_fit"],loaders["val_seen"],loaders["val_unseen"],
    candidates,verified_report,class_mappings,composition_mapping)
if exposure_table is not None:
    exposure_table.to_csv(experiment_dir/"expected_disease_sampling_exposure.csv",index=False)
print("FINISHED", EXPERIMENT_TO_RUN,"best epoch",metrics["epoch"],"saved to",experiment_dir)
del loaders,datasets,example,sampler,model
gc.collect()
plt.close("all")
if DEVICE.type == "cuda": torch.cuda.empty_cache()
print("Training closed; starting development-only audit. No second training job will start.")


## 8. Evaluate saved best checkpoint (development-only)

Use the same crop/disease class mappings and all 9,625 candidate pairs as Exp09. Saves validation predictions and error breakdown.


In [ ]:
if RUN_AUTO_AUDIT:
    ckpt = EXPERIMENT_DIR / "best_development.pt"
    if not ckpt.is_file():
        raise FileNotFoundError("Selected checkpoint missing; cannot audit")
    diagnostic_model, saved = load_checkpoint(ckpt, DEVICE)
    assert saved["stage"] == 1 and saved["config"]["experiment_id"] == EXPERIMENT_TO_RUN
    assert saved["development_split"]["sha256"] == split_report["sha256"], "Split mismatch"
    saved_candidates = torch.as_tensor(saved["candidate_pairs"], dtype=torch.long)
    train_pair_set = set(map(tuple, frames["train_fit"][["crop_id", "disease_id"]].to_numpy().tolist()))
    predictions, audit_summary = {}, []
    for split in ("val_seen", "val_unseen"):
        ds = DPDDataset(frames[split], IMAGE_ROOT, evaluation_transform(CONFIG["image_size"]))
        dl = make_loader(ds, saved["config"], shuffle=False)
        metrics_split, records = evaluate_loader(diagnostic_model, dl, DEVICE, saved_candidates,
                                                   amp=saved["config"]["amp"], collect_predictions=True)
        tab = pd.DataFrame(records)
        assert len(tab) == len(frames[split]) and tab.filename.is_unique
        tab["crop_correct"] = tab.crop_id == tab.predicted_crop_id
        tab["disease_correct"] = tab.disease_id == tab.predicted_disease_id
        tab["crop_only_wrong"] = ~tab.crop_correct & tab.disease_correct
        tab["disease_only_wrong"] = tab.crop_correct & ~tab.disease_correct
        tab["both_wrong"] = ~tab.crop_correct & ~tab.disease_correct
        tab["predicted_pair_seen_in_training"] = [
            (int(c), int(d)) in train_pair_set
            for c,d in zip(tab.predicted_pair_crop_id, tab.predicted_pair_disease_id)]
        if int((tab.crop_correct & tab.disease_correct).sum()) != round(metrics_split["composition_top1"] * len(tab)):
            print("Warning: independent head argmax and composition ranking differ (possible score ties).")
        predictions[split] = tab
        tab.to_csv(EXPERIMENT_DIR / f"{split}_error_predictions.csv", index=False)
        audit_summary.append({"split":split, "n":len(tab),
            "crop_top1":metrics_split["crop_top1"],
            "disease_top1":metrics_split["disease_top1"],
            "composition_top1":metrics_split["composition_top1"],
            "composition_top5":metrics_split["composition_top5"],
            "disease_only_wrong_count":int(tab.disease_only_wrong.sum()),
            "crop_only_wrong_count":int(tab.crop_only_wrong.sum()),
            "both_wrong_count":int(tab.both_wrong.sum()),
            "predicted_pair_seen_in_training_rate":float(tab.predicted_pair_seen_in_training.mean())})
    audit_table = pd.DataFrame(audit_summary)
    audit_table.to_csv(EXPERIMENT_DIR / "task_error_breakdown.csv",index=False)
    display(audit_table)
    print("Saved Experiment predictions + error breakdown")
else:
    print("Auto audit disabled. No evaluation performed.")


## 9. Per-class accuracy: minority disease diagnosis

Focus on `spider_mites`, `bacterial_blight`, `angular_leaf_spot` and whether larger disease weight improved disease-only mistakes and final composition HM.


In [ ]:
if RUN_AUTO_AUDIT:
    class_counts = {}
    for task in ("crop", "disease", "composition"):
        idcol, namecol = f"{task}_id", f"{task}_name"
        table = (frames["train_fit"].groupby([idcol, namecol], dropna=False)
                 .size().rename("train_images").reset_index()
                 .sort_values(["train_images",namecol], ascending=[True,True]))
        for split in ("val_seen", "val_unseen"):
            counts = frames[split].groupby(idcol).size().rename(f"{split}_images")
            table = table.join(counts,on=idcol)
        table[["val_seen_images","val_unseen_images"]] = table[["val_seen_images","val_unseen_images"]].fillna(0).astype(int)
        table.to_csv(EXPERIMENT_DIR / f"{task}_class_counts.csv",index=False)
        class_counts[task] = table
        print(f"{task}: {len(table)} labels | min {int(table.train_images.min())} | median {table.train_images.median():g} | max {int(table.train_images.max())}")

    MIN_SUPPORT = 5
    per_class = {}
    for split, pred in predictions.items():
        for task in ("crop", "disease"):
            idcol, namecol, predcol = f"{task}_id", f"{task}_name", f"predicted_{task}_id"
            rows = pred.assign(_correct=(pred[idcol]==pred[predcol]).astype(int))
            score = rows.groupby(idcol).agg(n_validation=(idcol,"size"), n_correct=("_correct","sum")).reset_index()
            score["top1_accuracy"] = score.n_correct / score.n_validation
            names = class_counts[task][[idcol,namecol,"train_images"]]
            score=score.merge(names,on=idcol,how="left",validate="one_to_one")
            score["n_wrong"] = score.n_validation - score.n_correct
            errs = rows.loc[rows._correct==0]
            frequent_wrong = (errs.groupby([idcol,predcol]).size().rename("mistake_count")
               .reset_index().sort_values("mistake_count",ascending=False).drop_duplicates(idcol))
            freq_names = dict(zip(names[idcol],names[namecol]))
            frequent_wrong["most_common_wrong_prediction"] = frequent_wrong[predcol].map(freq_names)
            score=score.merge(frequent_wrong[[idcol,"most_common_wrong_prediction","mistake_count"]],
                              on=idcol,how="left")
            score=score.sort_values(["top1_accuracy","n_validation"],ascending=[True,False])
            per_class[(split,task)]=score
            score.to_csv(EXPERIMENT_DIR / f"{split}_{task}_per_class.csv",index=False)
            eligible=score.loc[score.n_validation>=MIN_SUPPORT]
            print(f"\n{split} {task}: {len(eligible)} classes with n >= {MIN_SUPPORT}")
            display(eligible[[namecol,"train_images","n_validation","top1_accuracy","most_common_wrong_prediction"]].head(6))
            display(eligible.sort_values(["top1_accuracy","n_validation"],ascending=[False,False])
                    [[namecol,"train_images","n_validation","top1_accuracy","most_common_wrong_prediction"]].head(6))
else:
    print("Audit disabled")


## 10. Confusion matrices and weak/strong class visualizations

Same reporting method as previous Stage-1 experiments; figures saved in the new run folder.


In [ ]:
if RUN_AUTO_AUDIT:
    for split, pred in predictions.items():
        for task in ("crop","disease"):
            idcol, namecol, predcol = f"{task}_id", f"{task}_name", f"predicted_{task}_id"
            freq = class_counts[task]
            id_to_name=dict(zip(freq[idcol],freq[namecol]))
            all_ids=sorted(freq[idcol].unique().tolist())
            full = pd.crosstab(pred[idcol],pred[predcol]).reindex(index=all_ids,columns=all_ids,fill_value=0)
            full.index=[f"{i}: {id_to_name[i]}" for i in all_ids]
            full.columns=[f"{i}: {id_to_name[i]}" for i in all_ids]
            full.to_csv(EXPERIMENT_DIR / f"{split}_{task}_full_confusion.csv")
            qualified=per_class[(split,task)].query("n_validation >= @MIN_SUPPORT")
            selections={"weakest":qualified.head(6),
                        "strongest":qualified.sort_values(["top1_accuracy","n_validation"],ascending=[False,False]).head(6)}
            for label,selected in selections.items():
                chosen=selected[idcol].tolist()
                small=pred.loc[pred[idcol].isin(chosen),[idcol,predcol]].copy()
                common=(small.loc[small[idcol]!=small[predcol],predcol]
                        .value_counts().head(5).index.tolist())
                columns=["Correct"]+[f"{int(p)}: {id_to_name.get(int(p),p)}" for p in common]+["Other wrong"]
                counts, labels=[],[]
                for target in chosen:
                    part=small.loc[small[idcol]==target]
                    vals=[int((part[predcol]==target).sum())]
                    vals += [int(((part[predcol]==p)&(part[predcol]!=target)).sum()) for p in common]
                    vals.append(int(((part[predcol]!=target)&(~part[predcol].isin(common))).sum()))
                    assert sum(vals)==len(part), "Confusion row counts do not match support"
                    counts.append(vals)
                    labels.append(f"{id_to_name[target]} (n={len(part)})")
                matrix=pd.DataFrame(counts,index=labels,columns=columns)
                matrix.to_csv(EXPERIMENT_DIR / f"{split}_{task}_{label}_confusion_summary.csv")
                fig,ax=plt.subplots(figsize=(max(8,2+len(columns)*1.2),max(4, len(chosen)*0.6+1)))
                im=ax.imshow(matrix.to_numpy(), aspect="auto")
                ax.set_xticks(np.arange(len(columns)),labels=columns,rotation=45,ha="right",fontsize=8)
                ax.set_yticks(np.arange(len(labels)),labels=labels,fontsize=9)
                for (i,j),value in np.ndenumerate(matrix.to_numpy()):
                    ax.text(j,i,str(value),ha="center",va="center",fontsize=8)
                ax.set_title(f"{EXPERIMENT_TO_RUN} {split} {task} — {label} classes (counts)")
                fig.colorbar(im,ax=ax,label="Images")
                fig.tight_layout()
                fig.savefig(EXPERIMENT_DIR / f"{split}_{task}_{label}_confusion_summary.png",dpi=135,bbox_inches="tight")
                plt.close(fig)
    print("Saved all eight weak/strong confusion summary charts, CSVs and four full confusion matrices")
else:
    print("Audit disabled")


## 11. Per-seed development results and mean ± sample SD

Read Exp15 seed42 and available Exp18 checkpoints. Important: report the seeds and fixed development split. A difference of one correct unseen image changes unseen Top-1 by 0.25 percentage points. No official-test metrics.


In [ ]:
rows = []
for name,seed,path in [
    ("Exp15 original", 42, OUTPUT_ROOT / "exp15_disease_weight15"),
    ("Exp18 repeat", 123, OUTPUT_ROOT / "exp18_seed123"),
    ("Exp18 repeat", 2026, OUTPUT_ROOT / "exp18_seed2026"),
]:
    fp = path / "validation_metrics.json"
    if not fp.exists():
        print("Not yet completed:", fp)
        continue
    m = json.loads(fp.read_text(encoding="utf-8"))
    rows.append({"run":name,"seed":seed,"best_epoch":m["epoch"],
        "seen_composition":m["seen"]["composition_top1"],
        "unseen_composition":m["unseen"]["composition_top1"],
        "HM":m["composition_harmonic_mean"],
        "unseen_crop":m["unseen"]["crop_top1"],
        "unseen_disease":m["unseen"]["disease_top1"],
        "unseen_correct_of_400":round(m["unseen"]["composition_top1"]*400)})
results = pd.DataFrame(rows)
if results.empty: raise ValueError("No reference metrics found")
results.to_csv(EXPERIMENT_DIR / "exp15_exp18_multiseed_comparison.csv", index=False)
display(results)
metrics_cols=["seen_composition", "unseen_composition", "HM", "unseen_crop", "unseen_disease"]
summary=results[metrics_cols].agg(["mean", "std"]).rename(index={"std":"sample_std"})
summary.to_csv(EXPERIMENT_DIR / "seed_mean_std.csv")
display(summary)
all_done=len(results)==3
msg=["# Exp18 — Multi-Seed Robustness (Development Only)", "",
     "- Backbone: shared ResNet-50 V2, identical Exp15 recipe, disease task weight 1.5.",
     "- Fixed train/validation split; varied random seeds 42, 123, 2026 only.",
     "- Independent seed training results (not multiple dataset splits):", "",
     results.to_string(index=False), "",
     "- Mean / sample standard deviation over available seeds:", "",
     summary.to_string(), "",
     f"- All three seeds complete: {all_done}.",
     "- Development-only selection (checkpoint maximizes seen/unseen validation HM).",
     "- Stage 2 and official test NOT run. Do not tune further on official test.",
     "- One unseen correct prediction = 0.25 percentage points. Repeated development tuning may overfit validation.", ""]
(EXPERIMENT_DIR / "leader_multiseed_summary.md").write_text("\n".join(msg), encoding="utf-8")
print("Saved seed comparison; all seeds complete:", all_done)


## 12. Next steps

If `seed_mean_std.csv` shows wide variation, the tiny Exp15-vs-Exp10 improvement is not robustly established. If all three seeds are close, that is more support for the chosen training recipe (still not independent test validation). After Exp17 and Exp18, lock a model-selection rule and proceed to Stage 2 full-data fine-tuning, then official test.
